Anexo A, apartado A.4 · El ciclo completo: entrenar el adaptador y evaluar ensayo a ensayo.

Cuaderno que carga en una GPU de 80 GB la base Llama-3.1-70B en 4 bits con o sin el adaptador de Centaur
y publica un servicio que, para cada sesión recibida como texto, devuelve las probabilidades logarítmicas del token
siguiente en cada posición de elección, sin generar texto.


# Real Centaur 70B backend for ai-system-lab (Lightning, logprobs)

This notebook is intended to run in Lightning, Google Colab, or another GPU notebook and expose a real `/generate` endpoint for `ai-system-lab`.

This variant returns `response_tokens` and `response_token_logprobs` when `return_logprobs=True`, which is required for Nature-compatible NLL scoring.

Use a fresh kernel before running this notebook if a previous FastAPI server was already bound to port `8080`.


## 0. Persistent model cache + GPU memory hygiene

Punta el cache de Hugging Face al volumen persistente
`/teamspace/studios/this_studio/hf-cache` antes de cualquier import de HF/Unsloth.
La primera ejecucion descarga los pesos 4-bit (Unsloth mapea automaticamente
a `unsloth/...-bnb-4bit`, ~40-50 GB en total contando base + adapter); las
siguientes los reusan desde disco aunque pares y reinicies la GPU del Studio.

Tambien fija `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` para evitar
fragmentacion de VRAM y mitigar OOM en prompts largos (>32k chars). Sin esta
flag los runs con `collsioo2023MCPL` u otras tareas con prompts grandes
saltan CUDA out of memory hacia el medio del run.

Si NO estas en un Lightning Studio con ese path, cambia `CACHE_ROOT` a cualquier
directorio persistente que controles, o comenta las lineas `os.environ`.


In [ ]:
import os

CACHE_ROOT = "/teamspace/studios/this_studio/hf-cache"
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{CACHE_ROOT}/hub"
os.environ["TRANSFORMERS_CACHE"] = f"{CACHE_ROOT}/hub"
os.makedirs(f"{CACHE_ROOT}/hub", exist_ok=True)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("HF cache root:", CACHE_ROOT)
print("PYTORCH_CUDA_ALLOC_CONF:", os.environ["PYTORCH_CUDA_ALLOC_CONF"])


## 1. Check GPU

Run this first. If the available VRAM is below 80GB, do not continue with a real Centaur 70B run in this notebook.

In [ ]:
!nvidia-smi

import subprocess

def gpu_memory_mib():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
        capture_output=True,
        text=True,
        check=True,
    )
    return [int(x.strip()) for x in result.stdout.splitlines() if x.strip()]

memories = gpu_memory_mib()
print("GPU memory MiB:", memories)
if not memories or max(memories) < 76000:
    raise RuntimeError(
        "Centaur 70B real inference needs about 80GB VRAM for the adapter path. "
        "This runtime does not appear large enough. Use an A100/H100 80GB class runtime or a hosted endpoint."
    )


## 2. Install runtime dependencies (conditional)

La primera vez que arrancas el Studio instala unsloth, fastapi y huggingface_hub.
En arranques posteriores el conda env del Studio ya los tiene en disco, asi
que el bloque siguiente comprueba primero si estan importables y SE SALTA
el pip install completo si todo esta presente. Tipico ahorro: 30-90s por
arranque tras el primero.

Restart del runtime solo es necesario tras una instalacion fresca.


In [ ]:
import importlib, subprocess, sys

REQUIRED = ["unsloth", "unsloth_zoo", "fastapi", "uvicorn", "nest_asyncio", "requests", "huggingface_hub"]

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(mod)

if not missing:
    print("All runtime dependencies already installed, skipping pip install.")
else:
    print(f"Installing missing dependencies: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
        "git+https://github.com/unslothai/unsloth-zoo.git",
    ], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "fastapi", "uvicorn", "nest_asyncio", "requests", "huggingface_hub",
    ], check=True)
    print("Install complete.")


## 3. Authenticate with Hugging Face

Use a token that has access to the required Llama/Centaur model files.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()


## 4. Load real Centaur adapter

This is the real model path. It can take a long time and large disk space. If this cell fails for memory, the runtime is not sufficient.

In [ ]:
# Defensive env setup: if Step 0 was skipped, set the same vars here.
import os
CACHE_ROOT = "/teamspace/studios/this_studio/hf-cache"
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{CACHE_ROOT}/hub"
os.environ["TRANSFORMERS_CACHE"] = f"{CACHE_ROOT}/hub"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.makedirs(f"{CACHE_ROOT}/hub", exist_ok=True)
print("HF cache root:", CACHE_ROOT)
print("PYTORCH_CUDA_ALLOC_CONF:", os.environ["PYTORCH_CUDA_ALLOC_CONF"])

# Unsloth must be imported before transformers; do it here at the top.
from unsloth import FastLanguageModel

MODEL_NAME = "marcelbinz/Llama-3.1-Centaur-70B-adapter"
MAX_SEQ_LENGTH = 32768

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print("Loaded real Centaur adapter:", MODEL_NAME)


## 5. Local inference function

Centaur prompts should mark human choices with `<<` and `>>`, as recommended by the model card.

In [ ]:
import re
import torch
import warnings

warnings.filterwarnings(
    "ignore",
    message=".*AttentionMaskConverter.*",
    category=FutureWarning,
)

def _strip_prompt(decoded_text, prompt):
    return decoded_text[len(prompt):].strip() if decoded_text.startswith(prompt) else decoded_text.strip()

def _target_logprobs(prompt, target_text):
    if not target_text:
        return {}
    prompt_inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    full_inputs = tokenizer(prompt + target_text, return_tensors="pt").to("cuda")
    prompt_len = int(prompt_inputs["input_ids"].shape[-1])
    input_ids = full_inputs["input_ids"]
    target_ids = input_ids[0][prompt_len:]
    if len(target_ids) == 0:
        return {"target_text": target_text, "target_tokens": [], "target_token_logprobs": []}
    with torch.inference_mode():
        outputs = model(**full_inputs)
    log_probs = torch.log_softmax(outputs.logits[0], dim=-1)
    token_logprobs = []
    for absolute_pos in range(prompt_len, int(input_ids.shape[-1])):
        token_id = int(input_ids[0][absolute_pos])
        token_logprobs.append(float(log_probs[absolute_pos - 1, token_id].detach().cpu()))
    return {
        "target_text": target_text,
        "target_tokens": [
            tokenizer.decode([int(token_id)], skip_special_tokens=False)
            for token_id in target_ids.detach().cpu().tolist()
        ],
        "target_token_logprobs": token_logprobs,
    }

def _choice_spans(text):
    return [
        {
            "trial_index": index,
            "target_text": match.group(1).strip(),
            "start_char": match.start(1),
            "end_char": match.end(1),
        }
        for index, match in enumerate(re.finditer(r"<<\s*(.*?)\s*>>", text, flags=re.DOTALL))
    ]

def _per_trial_logprobs(text):
    spans = _choice_spans(text)
    if not spans:
        return []
    inputs = tokenizer(text, return_tensors="pt", return_offsets_mapping=True)
    offset_mapping = inputs.pop("offset_mapping")[0].tolist()
    inputs = inputs.to("cuda")
    with torch.inference_mode():
        outputs = model(**inputs)
    log_probs = torch.log_softmax(outputs.logits[0], dim=-1)
    input_ids = inputs["input_ids"][0]
    scored = []
    for span in spans:
        token_positions = [
            pos
            for pos, (start, end) in enumerate(offset_mapping)
            if end > span["start_char"] and start < span["end_char"] and pos > 0
        ]
        token_logprobs = []
        target_tokens = []
        for pos in token_positions:
            token_id = int(input_ids[pos])
            token_logprobs.append(float(log_probs[pos - 1, token_id].detach().cpu()))
            target_tokens.append(tokenizer.decode([token_id], skip_special_tokens=False))
        scored.append({**span, "target_tokens": target_tokens, "target_token_logprobs": token_logprobs})
    return scored

def centaur_generate(prompt, max_new_tokens=8, temperature=0.0, return_logprobs=False, target_text=None, return_per_trial_logprobs=False, per_trial_prompt=None):
    target_payload = _target_logprobs(prompt, target_text) if return_logprobs and target_text else {}
    per_trial_payload = (
        {"per_trial_logprobs": _per_trial_logprobs(per_trial_prompt or prompt)}
        if return_per_trial_logprobs
        else {}
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    generation_kwargs = {
        **inputs,
        "max_new_tokens": max_new_tokens,
        "do_sample": temperature > 0,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if temperature > 0:
        generation_kwargs["temperature"] = temperature
    if return_logprobs:
        generation_kwargs["return_dict_in_generate"] = True
        generation_kwargs["output_scores"] = True

    with torch.inference_mode():
        outputs = model.generate(**generation_kwargs)

    if not return_logprobs:
        decoded_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return {"generated_text": _strip_prompt(decoded_text, prompt), **target_payload, **per_trial_payload}

    sequence = outputs.sequences[0]
    input_len = inputs["input_ids"].shape[1]
    generated_ids = sequence[input_len:]
    transition_scores = model.compute_transition_scores(
        outputs.sequences,
        outputs.scores,
        normalize_logits=True,
    )[0]
    token_logprobs = [
        float(score)
        for score in transition_scores[: len(generated_ids)].detach().cpu().tolist()
    ]
    response_tokens = [
        tokenizer.decode([int(token_id)], skip_special_tokens=False)
        for token_id in generated_ids.detach().cpu().tolist()
    ]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return {
        "generated_text": generated_text,
        "response_tokens": response_tokens,
        "response_token_logprobs": token_logprobs,
        **target_payload,
        **per_trial_payload,
    }

test_prompt = "In this experiment, a participant must choose between <<A>> and <<B>>. The participant chooses <<"
print(centaur_generate(test_prompt, max_new_tokens=8, temperature=0.0, return_logprobs=True, target_text="A"))


## 6. Expose `/generate` inside Colab

This starts a FastAPI app on port 8080 inside the Colab runtime.

In [ ]:
import nest_asyncio
import socket
import threading
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel, Field

nest_asyncio.apply()

API_PORT = 8080

def _port_is_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex(("127.0.0.1", port)) == 0

if _port_is_open(API_PORT):
    raise RuntimeError(
        f"Port {API_PORT} is already in use. Restart the notebook kernel/runtime "
        "or stop the previous FastAPI server before running this cell."
    )

app = FastAPI(title="Real 70B Lightning Backend")

class GenerateRequest(BaseModel):
    prompt: str
    model: str = MODEL_NAME
    max_new_tokens: int = Field(default=8, ge=1, le=256)
    temperature: float = Field(default=0.0, ge=0.0, le=2.0)
    return_logprobs: bool = False
    target_text: str | None = None
    return_per_trial_logprobs: bool = False
    per_trial_prompt: str | None = None

@app.get("/health")
def health():
    return {
        "status": "ok",
        "ready": True,
        "backend": "colab-unsloth",
        "model": MODEL_NAME,
        "synthetic_fallback_allowed": False,
        "supports_response_token_logprobs": True,
        "supports_target_token_logprobs": True,
        "supports_per_trial_logprobs": True,
    }

@app.post("/generate")
def generate(req: GenerateRequest):
    result = centaur_generate(
        req.prompt,
        max_new_tokens=req.max_new_tokens,
        temperature=req.temperature,
        return_logprobs=req.return_logprobs,
        target_text=req.target_text,
        return_per_trial_logprobs=req.return_per_trial_logprobs,
        per_trial_prompt=req.per_trial_prompt,
    )
    response = {
        "model": req.model,
        "generated_text": result["generated_text"],
        "synthetic": False,
        "backend": "colab-unsloth",
    }
    if req.return_logprobs:
        response["response_tokens"] = result.get("response_tokens", [])
        response["response_token_logprobs"] = result.get("response_token_logprobs", [])
        response["target_text"] = result.get("target_text")
        response["target_tokens"] = result.get("target_tokens", [])
        response["target_token_logprobs"] = result.get("target_token_logprobs", [])
    if req.return_per_trial_logprobs:
        response["per_trial_logprobs"] = result.get("per_trial_logprobs", [])
    return response

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=API_PORT, log_level="info")

thread = threading.Thread(target=run_api, daemon=True)
thread.start()
print(f"Lightning-local API running on http://127.0.0.1:{API_PORT}")


## 7. Smoke-test the local API

Run this before opening the public tunnel. It verifies that `/health` and `/generate` work inside the GPU runtime and that `response_token_logprobs` is returned.


In [ ]:
import requests

health_response = requests.get("http://127.0.0.1:8080/health", timeout=30)
print("health", health_response.status_code, health_response.text[:1000])
health_response.raise_for_status()

generate_response = requests.post(
    "http://127.0.0.1:8080/generate",
    json={
        "prompt": "In this experiment, a participant must choose between <<A>> and <<B>>. The participant chooses <<",
        "target_text": "A",
        "return_per_trial_logprobs": True,
        "per_trial_prompt": "Trial 1: participant chooses <<A>>. Trial 2: participant chooses <<B>>.",
        "max_new_tokens": 8,
        "temperature": 0.0,
        "return_logprobs": True,
    },
    timeout=120,
)
print("generate", generate_response.status_code, generate_response.text[:1000])
generate_response.raise_for_status()
generate_payload = generate_response.json()
assert "response_token_logprobs" in generate_payload, generate_payload
assert "target_token_logprobs" in generate_payload, generate_payload
assert "per_trial_logprobs" in generate_payload, generate_payload
assert isinstance(generate_payload["target_token_logprobs"], list), generate_payload
assert generate_payload["target_token_logprobs"], generate_payload
assert len(generate_payload["per_trial_logprobs"]) == 2, generate_payload
print("logprobs_ok", len(generate_payload["response_token_logprobs"]), "target_logprobs_ok", len(generate_payload["target_token_logprobs"]), "per_trial_ok", len(generate_payload["per_trial_logprobs"]))


## 8. Public tunnel for your local repo

Use cloudflared to expose the Colab API. Copy the printed `https://...trycloudflare.com` URL and use it as `CENTAUR_UPSTREAM_URL` locally.

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
!./cloudflared tunnel --url http://127.0.0.1:8080 --no-autoupdate


## 9. Configure ai-system-lab locally

On your Windows machine, keep the GPU notebook and Cloudflare tunnel running. Copy the `https://...trycloudflare.com` URL and run:

```powershell
cd <REPO>
powershell -ExecutionPolicy Bypass -File ops\configure_centaur_colab_upstream.ps1 -TunnelUrl "https://YOUR-COLAB-TUNNEL.trycloudflare.com" -LocalPort 18080
powershell -ExecutionPolicy Bypass -File ops\start_centaur_service.ps1 -Port 18080
python ops\test_centaur_endpoint.py --base-url http://127.0.0.1:18080 --timeout-s 300
python examples\run_real_centaur_psych101_case.py --input-path datasets\psych101\psych101_eval_stratified_16.jsonl --limit 16 --max-observed-choices 64 --timeout-s 300 --run-id stratified16-window64-a100
```

Use local port `18080` unless `8080` is free. On the current Windows machine, `8080` was occupied by `AgentService`.
